# EDA — Fenrir v2.1

**Dataset:** [`AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1`](https://huggingface.co/datasets/AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1)

≈99 870 tripletas system/user/assistant de ciberseguridad defensiva (OWASP, MITRE ATT&CK, NIST CSF, CIS…) para SFT.

**Formato:** JSON Lines (`220426-CyberSec-Dataset_escaped.jsonl`). Columnas: `system`, `user`, `assistant`.

**Preguntas guía para el EDA**
- ¿Cuántos system prompts distintos hay?
- ¿Hay preguntas o respuestas duplicadas?
- ¿Qué tan frecuentes son los rechazos que promete el README?
- ¿Se solapa con Trendyol (#06)? (ver notebook 06)

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Busca la raíz del repo (donde está src/profiler.py) y la agrega al path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "profiler.py").exists())
sys.path.insert(0, str(ROOT))
from src.profiler import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

ctx = eda_context("05")   # rutas: datos en data/raw/..., resultados en esta carpeta
print(ctx)

## 1. Inventario de archivos

In [ ]:
files = list_files(ctx.data_dir)
display(summarize_extensions(files))
display(files)

## 2. Cargar (muestra)
El archivo es grande, así que por defecto se toma una **muestra aleatoria** de `SAMPLE_N` filas
(se recorre el archivo completo pero solo se guarda la muestra en memoria).
Cambia `FULL = True` para cargarlo entero si tu equipo tiene RAM suficiente.

In [ ]:
PATH = ctx.data_dir / "220426-CyberSec-Dataset_escaped.jsonl"
SAMPLE_N = 20_000
FULL = False

print("Primer registro:"); display(peek_record(PATH))
N_TOTAL = count_lines(PATH)
print(f"Filas totales: {N_TOTAL:,}")

df = read_jsonl(PATH) if FULL else read_jsonl(PATH, sample=SAMPLE_N)
print("Filas cargadas:", len(df))
display(df.head(3))
display(profile_df(df))

## 3. Largo de los textos

In [ ]:
L = text_lengths(df, ["system", "user", "assistant"])
display(L.describe().round(1))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, c in zip(axes, L.columns):
    sns.histplot(L[c].clip(lower=1), bins=50, log_scale=True, ax=ax); ax.set_title(f"{c} (caracteres, log)")
plt.tight_layout(); save_fig(fig, ctx, "largo_textos"); plt.show()

## 4. Prompts de sistema, duplicados y plantillas

In [ ]:
print("System prompts distintos:", df["system"].nunique())
display(df["system"].str[:150].value_counts().head(10).to_frame("n"))

display(duplicates_report(df, {"user": ["user"], "assistant": ["assistant"], "user + assistant": ["user", "assistant"]}))

# Señal de plantillas: primeras 4 palabras más comunes del mensaje del usuario
first_words = df["user"].fillna("").str.split().str[:4].str.join(" ")
display(first_words.value_counts().head(15).to_frame("n"))

In [ ]:
# ¿Cuántas respuestas son negativas/rechazos? (patrón aproximado)
REFUSAL = r"(?i)\b(?:I can(?:no|')t|I won't|I'm sorry|I am sorry|I must decline|I can't assist)\b"
ref = df["assistant"].fillna("").str.contains(REFUSAL)
print(f"Respuestas con patrón de rechazo: {ref.sum():,} ({ref.mean():.1%})")

## Ejemplos

In [ ]:
show_examples(df, n=2, cols=["user", "assistant"], max_chars=800)

## Guardar resultados
Se guardan en `tables/` de esta carpeta. `resumen.csv` lo consolida `eda/00_general`.

In [ ]:
save_table(profile_df(df), ctx, "perfil_columnas")
display(save_summary(ctx, "AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1", df, total_rows=N_TOTAL, dup_cols=["user", "assistant"],
                     notas="análisis sobre muestra aleatoria"))

## Próximos pasos / notas
- Anota tus hallazgos en el `README.md` de esta carpeta.
-